In [1]:
setwd("/well/lindgren-ukbb/projects/ukbb-11867/flassen/projects/KO/wes_ko_ukbb")
devtools::load_all('utils/modules/R/prstools')
library(argparse)

i Loading PRStools

Loading required package: bigsnpr

Loading required package: bigstatsr

Loading required package: data.table

Loading required package: bigassertr



In [2]:
args <- list(
    pred = "data/prs/hapmap/ukb_500k/ukb_hapmap_500k_eur_chr21.bed",
    ldsc = "data/prs/ldsc/ldsc_Alanine_aminotransferase_residual.rds",
    ld_dir = "data/prs/hapmap/ld/matrix",
    method = "inf",
    impute = "mean2",
    out_prefix = "data/prs/scores/auto/prs_auto_Alanine_aminotransferase_residual_chr21"
)


In [3]:
args$tmp_bfile <- paste0(args$out_prefix,".bfile")
args$chrom <- "chr21"

In [4]:
stopifnot(file.exists(args$pred))
stopifnot(file.exists(args$ldsc))
stopifnot(!file.exists(args$tmp_bfile))
stopifnot(dir.exists(args$ld_dir))
stopifnot(args$chrom %in% paste0("chr",1:22))
stopifnot(dir.exists(dirname(args$out_prefix)))
stopifnot(args$method %in% c('inf', 'auto'))

In [5]:
# setup parallel environment
NCORES <- max(1, nb_cores())
bigparallelr::assert_cores(NCORES)

# laod ldsc and qced GWAS
ldsc <- readRDS(args$ldsc)
gwas <- ldsc$gwas
h2 <- ldsc$coefficients$estimate[2]
pvalue <- ldsc$coefficients$pvalue[2]

# Estimate h2 chromosome-wide
N_total <- nrow(gwas)
N_chr <- sum(gwas$chr == args$chrom)
h2_init <- h2 * (N_chr / N_total)

# check estimates and ensure that
# the trait is actually heritable
stopifnot(pvalue < 1e-5)
stopifnot(!is.null(gwas))

# get SNP correlations and LD
snp <- get_single_ld_matrix(gwas, chr = args$chrom, ld_dir = args$ld_dir)

# match GWAS with snp-correlation map
gwas <- gwas[na.omit(match(snp$map$marker, gwas$marker)), ]

# check that LD-matrix markers and gwas markers have overlap
# Check that ordering of markers are actually matching
stopifnot(all(gwas$marker %in% snp$map$marker))
stopifnot(all(snp$map$marker %in% gwas$marker))
stopifnot(sum(gwas$marker == snp$map$marker) / nrow(gwas) == 1)


In [8]:
bfile <- tempfile(tmpdir = dirname(args$out_prefix))
pred <- load_bigsnp_from_bed(args$pred)

In [20]:
sd(pred$G[,1], na.rm = TRUE)

[1] 0.3080684

In [ ]:
# load data to be used for prediction

pred <- match_bigsnp_with_gwas(obj=pred, gwas=gwas, bfile=args$tmp_bfile)
genotypes <- pred$genotypes
indicies <- pred$gwas_indicies

In [ ]:
# check that we have genotypes
stopifnot(!is.null(genotypes))
stopifnot(!is.null(indicies))

# dimensions
cols <- genotypes$`.->ncol`
rows <- genotypes$`.->nrow`
missing_gt <- NA

# need to impute missing SNPs
if (!is.null(args$impute)){
sum_rows <- lapply(1:rows, function(i) return(sum(is.na(genotypes[i,]))))
missing_gt <- sum(unlist(sum_rows))
genotypes <- snp_fastImputeSimple(genotypes, method = args$impute)
}


In [13]:
# standardize genotypes
means <- NULL; sds <- NULL
if (args$standardized_gt){
 means <- as.numeric(unlist(lapply(1:rows, mean)))
 sds <- as.numeric(unlist(lapply(1:rows, sd)))
 write(means, paste0(args$out_prefix, ".means"))
 write(sds, paste0(args$out_prefix, ".sds"))
}

ERROR: Error in if (args$standardized_gt) {: argument is of length zero


In [ ]:
if (args$method %in% "inf"){

beta_inf <- snp_ldpred2_inf(snp$corr, gwas, h2_init)
beta_inf <- beta_inf[indicies]
final_pred_inf <- big_prodVec(
      genotypes,
      beta_inf,
      center = means,
      scale = sds,
      ncores = NCORES)
final_pred <- final_pred_inf
    
    }

In [ ]:

if (args$method %in% "inf"){

beta_inf <- snp_ldpred2_inf(snp$corr, gwas, h2_init)
beta_inf <- beta_inf[indicies]
final_pred_inf <- big_prodVec(
      genotypes,
      beta_inf,
      center = means,
      scale = sds,
      ncores = NCORES)
final_pred <- final_pred_inf

} else if (args$method %in% "auto") {

 write("Running multi_auto", stderr())
 multi_auto <- snp_ldpred2_auto(
    corr = snp$corr,
    df_beta = gwas,
    h2_init = h2_init,
    vec_p_init = seq_log(1e-4, 0.9, 30),
    ncores = NCORES)

 # save data chains
 multi_auto_path <- paste0(args$out_prefix,'_chains.rda')
 saveRDS(multi_auto, multi_auto_path, compress = 'xz')

 # get estimates with indicies corresponding to pred genotypes
 beta_auto <- sapply(multi_auto, function(auto){
      auto$beta_est})

 # perform matrix multiplication
 pred_auto <- big_prodMat(
    genotypes,
    beta_auto,
    center = means,
    scale = sds,
    ncores = NCORES)

 # quality controls on chains
 sc <- apply(pred_auto, 2, sd, na.rm = TRUE)
 keep <- abs(sc - median(sc)) < 3 * mad(sc)
 final_beta_auto <- rowMeans(beta_auto[, keep], na.rm = TRUE)

 # get final predicton
 final_pred_auto <- big_prodVec(
   genotypes,
   final_beta_auto,
   center = means,
   scale = sds,
   ncores = NCORES)

 final_pred <- final_pred_auto
}


# save parameters
model <- data.table(
 method = args$method,
 n_samples = unique(gwas$n),
 n_eff = unique(gwas$n_eff),
 n_snps = nrow(gwas),
 ldsc_h2_genome_wide_est = h2,
 h2_init_est = h2_init,
 inflation = calc_inflation(gwas$P),
 standardized_gt = args$standardized_gt,
 gt_rows = rows,
 gt_cols = cols,
 missing_gt = missing_gt
 )

# save results
PGS <- data.table(
sid = pred$sid,
prs = final_pred
)